In [ ]:
from pathlib import Path
import pandas as pd
import re
import uuid
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import os
from google import genai
from google.genai import types
import docx
import pymupdf





In [ ]:

DATA_DIR = Path("../sources")  # run from server/knowledge_base/notebooks/
BUILD_DIR = Path("../processed")
DOMAIN_MAPPING = {
    "Pháp lý & Quy định nhà nước": "legal_policy",
    "Hợp đồng": "legal_policy",
    "Bảng giá & Chính sách thương mại": "commercial_cs",
    "Chính sách thanh toán & Dịch vụ": "commercial_cs",
    "Nội quy BQL & Chủ đầu tư": "operations_sop",
    "Nghiệp vụ xử lý sự cố (SOP & FAQ)": "operations_sop",
    "Tiêu chuẩn kỹ thuật & Hạ tầng": "technical_infra",
}

records = [
    {
        "folder_name": (folder := item.relative_to(DATA_DIR).parts[0] if len(item.relative_to(DATA_DIR).parts) > 1 else "ROOT"),
        "domain": DOMAIN_MAPPING.get(folder, "other"),
        "file_name": item.name,
        "extension": item.suffix.lower(),
        "size_kb": round(item.stat().st_size / 1024, 2),
        "file_path": str(item)
    }
    for item in DATA_DIR.rglob("*")
    if item.is_file() and not item.name.startswith("~$")
]

df_manifest = pd.DataFrame(records)
df_manifest.to_csv(BUILD_DIR / "manifest.csv", index=False, encoding="utf-8-sig")

print(f"✅ Đã quét xong {len(df_manifest)} files hợp lệ:")
display(pd.crosstab(df_manifest["folder_name"], df_manifest["extension"], margins=True))

✅ Đã quét xong 15 files hợp lệ:


extension,.doc,.docx,.md,.pdf,.xlsx,All
folder_name,,,,,,
Bảng giá & Chính sách thương mại,0,0,0,0,1,1
Chính sách thanh toán & Dịch vụ,0,1,0,0,0,1
Hợp đồng,2,0,0,0,0,2
Nghiệp vụ xử lý sự cố (SOP & FAQ),0,0,0,1,0,1
Nội quy BQL & Chủ đầu tư,0,3,0,0,0,3
Pháp lý & Quy định nhà nước,3,0,1,1,0,5
Tiêu chuẩn kỹ thuật & Hạ tầng,0,1,0,1,0,2
All,5,5,1,3,1,15


In [ ]:


df_manifest = pd.read_csv(BUILD_DIR / "manifest.csv")

def parse_docx(file_path: Path) -> str:
    doc = docx.Document(file_path)
    output = []
    for block in doc.element.body:
        if block.tag.endswith('p'):
            text = "".join(block.itertext()).strip()
            if text:
                output.append(text)
        elif block.tag.endswith('tbl'):
            for table in doc.tables:
                if table._element is block:
                    rows = [[cell.text.strip().replace("\n", " ") for cell in row.cells] for row in table.rows]
                    if rows:
                        header = "| " + " | ".join(rows[0]) + " |"
                        sep = "| " + " | ".join(["---"] * len(rows[0])) + " |"
                        body = "\n".join(["| " + " | ".join(r) + " |" for r in rows[1:]])
                        output.append(f"\n{header}\n{sep}\n{body}\n")
                    break
    return "\n\n".join(output)

def parse_pdf_or_doc(file_path: Path) -> str:
    # 1. Thử đọc bằng PyMuPDF (xử lý được PDF gốc & file .doc tải từ web dạng PDF/XML)
    try:
        with pymupdf.open(str(file_path)) as doc:
            pages = [page.get_text("text") for page in doc]
        text = "\n\n".join(pages).strip()
        if text:
            return text
    except Exception:
        pass
    
    # 2. Dự phòng nếu là file .docx đổi đuôi thành .doc
    try:
        return parse_docx(file_path)
    except Exception:
        pass

    # 3. Dự phòng cuối nếu file .doc bên trong là HTML/MHT tải từ trang luật
    raw_html = file_path.read_text(encoding="utf-8", errors="ignore")
    clean_text = re.sub(r"<[^>]+>", " ", raw_html)
    return re.sub(r"\s+", " ", clean_text).strip()

def parse_xlsx(file_path: Path) -> str:
    xls = pd.ExcelFile(file_path)
    sheets_text = []
    for sheet_name in xls.sheet_names:
        df = pd.read_excel(xls, sheet_name=sheet_name).dropna(how="all")
        if df.empty:
            continue
        row_lines = [f"### Bảng dữ liệu (Sheet: {sheet_name})"]
        for _, row in df.iterrows():
            pairs = [f"{col}: {val}" for col, val in row.items() if pd.notna(val) and str(val).strip() != ""]
            if pairs:
                row_lines.append("- " + " | ".join(pairs))
        sheets_text.append("\n".join(row_lines))
    return "\n\n".join(sheets_text)

def extract_content(row) -> str:
    path = Path(row["file_path"])
    ext = row["extension"]
    try:
        if ext == ".md":
            return path.read_text(encoding="utf-8")
        elif ext == ".docx":
            return parse_docx(path)
        elif ext in [".pdf", ".doc"]:
            return parse_pdf_or_doc(path)
        elif ext in [".xlsx", ".xls"]:
            return parse_xlsx(path)
        return ""
    except Exception as e:
        print(f"❌ Lỗi đọc file {path.name}: {e}")
        return ""

df_manifest["raw_markdown"] = df_manifest.apply(extract_content, axis=1)
df_manifest["char_count"] = df_manifest["raw_markdown"].str.len()
df_manifest.to_json(BUILD_DIR / "parsed_documents.jsonl", orient="records", lines=True, force_ascii=False)

print("✅ Hoàn tất bóc tách nội dung!")
display(df_manifest[["folder_name", "file_name", "extension", "char_count"]])

✅ Hoàn tất bóc tách nội dung!


,folder_name,file_name,extension,char_count
0,Bảng giá & Chính sách thương mại,Bang_Gia_Do_Xe_O_To_HaNoi_TPHCM.xlsx,.xlsx,15992
1,Chính sách thanh toán & Dịch vụ,"Quy chế quản lý, thanh toán.docx",.docx,49045
2,Hợp đồng,hop-dong-gui-giu-xe-o-to_0105111058.doc,.doc,15123
3,Hợp đồng,Mẫu hợp đồng cung cấp dịch vụ giữ xe.doc,.doc,39370
4,Nghiệp vụ xử lý sự cố (SOP & FAQ),Cẩm nang Xử lý Sự cố & Kịch bản CSKH Bãi đỗ xe...,.pdf,12136
5,Nội quy BQL & Chủ đầu tư,Bai_Do_Xe_Thong_Minh_Su_Dung_The_Tu.docx,.docx,18596
6,Nội quy BQL & Chủ đầu tư,Noi_Quy_Bai_Do_Xe_Chuan_Van_Ban_Phap_Luat.docx,.docx,38222
7,Nội quy BQL & Chủ đầu tư,Quy định về hoạt động của bãi giữ xe.docx,.docx,4678
8,Pháp lý & Quy định nhà nước,102217_tt24bgtvt.doc,.doc,105756
9,Pháp lý & Quy định nhà nước,GIÁ DỊCH VỤ TRÔNG GIỮ.doc,.doc,47555


In [ ]:


df_docs = pd.read_json(BUILD_DIR / "parsed_documents.jsonl", lines=True)
df_docs = df_docs[df_docs["char_count"] > 0].copy()

def enrich_metadata(row):
    fname = row["file_name"].lower()
    folder = row["folder_name"]
    
    if "ha_noi" in fname or "hà nội" in fname or "smart city" in fname:
        city = "HN"
    elif "tphcm" in fname or "hồ chí minh" in fname or "grand park" in fname:
        city = "HCM"
    else:
        city = "ALL"
        
    sub_domain_map = {
        "Pháp lý & Quy định nhà nước": "state_law",
        "Hợp đồng": "contract_template",
        "Bảng giá & Chính sách thương mại": "pricing_policy",
        "Chính sách thanh toán & Dịch vụ": "billing_refund",
        "Nội quy BQL & Chủ đầu tư": "building_rules",
        "Nghiệp vụ xử lý sự cố (SOP & FAQ)": "incident_sop",
        "Tiêu chuẩn kỹ thuật & Hạ tầng": "technical_spec",
    }
    return pd.Series([sub_domain_map.get(folder, "general"), city])

df_docs[["sub_domain", "city"]] = df_docs.apply(enrich_metadata, axis=1)

def chunk_legal_or_policy(text: str, max_chars=2500):
    sections = re.split(r'(?=\n(?:Điều\s+\d+[\.:]|Chương\s+[IVX0-9]+|PHẦN\s+[IVX0-9]+))', text)
    chunks, current_chunk = [], ""
    for sec in sections:
        sec = sec.strip()
        if not sec:
            continue
        if len(current_chunk) + len(sec) <= max_chars:
            current_chunk = (current_chunk + "\n\n" + sec).strip()
        else:
            if current_chunk:
                chunks.append(current_chunk)
            if len(sec) > max_chars:
                chunks.extend([sec[i:i+max_chars] for i in range(0, len(sec), max_chars - 300)])
                current_chunk = ""
            else:
                current_chunk = sec
    if current_chunk:
        chunks.append(current_chunk)
    return chunks

def chunk_by_headings_or_size(text: str, max_chars=2000, overlap=250):
    blocks = re.split(r'(?=\n(?:#{1,3}\s+|\d+\.\s+[A-ZÀÁÂÃÈÉÊÌÍÒÓÔÕÙÚĂĐĨŨƠƯ]))', text)
    chunks, current_chunk = [], ""
    for blk in blocks:
        blk = blk.strip()
        if not blk:
            continue
        if len(current_chunk) + len(blk) <= max_chars:
            current_chunk = (current_chunk + "\n\n" + blk).strip()
        else:
            if current_chunk:
                chunks.append(current_chunk)
            if len(blk) > max_chars:
                for i in range(0, len(blk), max_chars - overlap):
                    chunks.append(blk[i:i+max_chars])
                current_chunk = ""
            else:
                current_chunk = blk
    if current_chunk:
        chunks.append(current_chunk)
    return chunks

rag_chunks = []
for _, row in df_docs.iterrows():
    domain = row["domain"]
    text = row["raw_markdown"]
    
    if domain in ["legal_policy"] or row["sub_domain"] in ["building_rules", "billing_refund"]:
        raw_chunks = chunk_legal_or_policy(text, max_chars=2500)
    else:
        raw_chunks = chunk_by_headings_or_size(text, max_chars=2000, overlap=250)
        
    for idx, ch_text in enumerate(raw_chunks, start=1):
        contextual_text = f"[Nguồn: {row['file_name']} | Nhóm: {row['folder_name']}]\n{ch_text}"
        rag_chunks.append({
            "chunk_id": f"{row['domain'].upper()}_{uuid.uuid4().hex[:8]}",
            "text": contextual_text,
            "metadata": {
                "domain": row["domain"],
                "sub_domain": row["sub_domain"],
                "folder_name": row["folder_name"],
                "source_file": row["file_name"],
                "city": row["city"],
                "chunk_index": idx,
                "total_chunks": len(raw_chunks),
                "status": "active"
            }
        })

df_chunks = pd.DataFrame(rag_chunks)
df_chunks.to_json(BUILD_DIR / "chunks_v1.jsonl", orient="records", lines=True, force_ascii=False)

print(f"🎉 HOÀN TẤT! Đã đóng gói {len(df_chunks)} chunks chuẩn hóa tại: processed/chunks_v1.jsonl\n")
meta_df = pd.json_normalize(df_chunks["metadata"])
display(pd.crosstab(meta_df["domain"], meta_df["folder_name"], margins=True))

🎉 HOÀN TẤT! Đã đóng gói 412 chunks chuẩn hóa tại: processed/chunks_v1.jsonl



folder_name,Bảng giá & Chính sách thương mại,Chính sách thanh toán & Dịch vụ,Hợp đồng,Nghiệp vụ xử lý sự cố (SOP & FAQ),Nội quy BQL & Chủ đầu tư,Pháp lý & Quy định nhà nước,Tiêu chuẩn kỹ thuật & Hạ tầng,All
domain,,,,,,,,
commercial_cs,11,30,0,0,0,0,0,41
legal_policy,0,0,25,0,0,249,0,274
operations_sop,0,0,0,9,34,0,0,43
technical_infra,0,0,0,0,0,0,54,54
All,11,30,25,9,34,249,54,412


In [ ]:

# 1. Nạp lại dữ liệu bóc tách ở Bước 2
df_docs = pd.read_json(BUILD_DIR / "parsed_documents.jsonl", lines=True)
df_docs = df_docs[df_docs["char_count"] > 0].copy()

sub_domain_map = {
    "Pháp lý & Quy định nhà nước": "state_law",
    "Hợp đồng": "contract_template",
    "Bảng giá & Chính sách thương mại": "pricing_policy",
    "Chính sách thanh toán & Dịch vụ": "billing_refund",
    "Nội quy BQL & Chủ đầu tư": "building_rules",
    "Nghiệp vụ xử lý sự cố (SOP & FAQ)": "incident_sop",
    "Tiêu chuẩn kỹ thuật & Hạ tầng": "technical_spec",
}

def smart_chunking(text: str, is_legal: bool, max_chars=1800, overlap=250):
    # Nhận diện ranh giới Điều/Chương kể cả khi file .doc bị mất ký tự xuống dòng
    if is_legal:
        sections = re.split(r'(?=(?:\n|\s)(?:Điều\s+\d+[\.:]|Chương\s+[IVX0-9]+|PHẦN\s+[IVX0-9]+))', text)
    else:
        sections = re.split(r'(?=\n(?:#{1,3}\s+|\d+\.\s+[A-ZÀÁÂÃÈÉÊÌÍÒÓÔÕÙÚĂĐĨŨƠƯ]))', text)
        
    chunks, current_chunk = [], ""
    for sec in sections:
        sec = re.sub(r'\s+', ' ', sec).strip() if "\n" not in sec else sec.strip()
        if not sec:
            continue
        if len(current_chunk) + len(sec) <= max_chars:
            current_chunk = (current_chunk + "\n\n" + sec).strip()
        else:
            if current_chunk:
                chunks.append(current_chunk)
            if len(sec) > max_chars:
                for i in range(0, len(sec), max_chars - overlap):
                    chunks.append(sec[i:i+max_chars])
                current_chunk = ""
            else:
                current_chunk = sec
    if current_chunk:
        chunks.append(current_chunk)
    return chunks

def detect_chunk_city(text: str) -> str:
    t = text.lower()
    has_hn = any(k in t for k in ["hà nội", "vành đai i", "vành đai ii", "vành đai iii", "smart city", "ocean park"])
    has_hcm = any(k in t for k in ["hồ chí minh", "tp.hcm", "tphcm", "grand park", "quận 1", "thủ đức"])
    if has_hn and not has_hcm:
        return "HN"
    if has_hcm and not has_hn:
        return "HCM"
    return "ALL"

# 2. Tái tạo lại rag_chunks_v1.jsonl chuẩn xác hơn
rag_chunks = []
for _, row in df_docs.iterrows():
    domain = row["domain"]
    sub_domain = sub_domain_map.get(row["folder_name"], "general")
    is_legal = domain == "legal_policy" or sub_domain in ["building_rules", "billing_refund"]
    
    raw_chunks = smart_chunking(row["raw_markdown"], is_legal=is_legal, max_chars=1800, overlap=250)
    for idx, ch_text in enumerate(raw_chunks, start=1):
        contextual_text = f"[Nguồn: {row['file_name']} | Nhóm: {row['folder_name']}]\n{ch_text}"
        rag_chunks.append({
            "chunk_id": f"{domain.upper()}_{uuid.uuid4().hex[:8]}",
            "text": contextual_text,
            "metadata": {
                "domain": domain,
                "sub_domain": sub_domain,
                "folder_name": row["folder_name"],
                "source_file": row["file_name"],
                "city": detect_chunk_city(ch_text),
                "chunk_index": idx,
                "total_chunks": len(raw_chunks),
                "status": "active"
            }
        })

df_chunks = pd.DataFrame(rag_chunks)
df_chunks.to_json(BUILD_DIR / "chunks_v1.jsonl", orient="records", lines=True, force_ascii=False)

meta_df = pd.json_normalize(df_chunks["metadata"])
df_rag = pd.concat([df_chunks[["chunk_id", "text"]], meta_df], axis=1)

# 3. Xây dựng lại Index & Hybrid Retriever với Soft Domain Boost
def tokenize_vi(text: str):
    return re.sub(r"[^\w\s]", " ", text.lower()).split()

bm25_index = BM25Okapi([tokenize_vi(doc) for doc in df_rag["text"]])
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=20000)
tfidf_matrix = tfidf_vectorizer.fit_transform(df_rag["text"])

def retrieve_chunks_v2(query: str, top_k: int = 2, preferred_domain: str = None, city_filter: str = None):
    mask = pd.Series([True] * len(df_rag))
    if city_filter:
        mask &= df_rag["city"].isin([city_filter, "ALL"])
    valid_indices = df_rag[mask].index.tolist()

    bm25_scores = np.array(bm25_index.get_scores(tokenize_vi(query)))
    if bm25_scores.max() > 0:
        bm25_scores = bm25_scores / bm25_scores.max()

    cos_scores = cosine_similarity(tfidf_vectorizer.transform([query]), tfidf_matrix).flatten()
    
    # Hybrid Score + Soft Boost (+0.15) nếu khớp domain dự kiến (không loại bỏ tài liệu liên ngành)
    domain_boost = np.where(df_rag["domain"] == preferred_domain, 0.15, 0.0) if preferred_domain else 0.0
    final_scores = 0.55 * bm25_scores + 0.45 * cos_scores + domain_boost

    ranked_indices = sorted(valid_indices, key=lambda idx: final_scores[idx], reverse=True)[:top_k]
    results = df_rag.loc[ranked_indices].copy()
    results["score"] = [round(final_scores[idx], 4) for idx in ranked_indices]
    return results

# 4. Chạy kiểm thử lại đúng 4 câu hỏi trên
test_queries = [
    {"query": "Giá gửi xe ô tô dưới 9 chỗ theo tháng ở khu vực Vành đai I Hà Nội", "domain": "commercial_cs", "city": "HN"},
    {"query": "Xử lý thế nào khi khách hàng làm mất thẻ từ gửi xe hoặc gặp sự cố?", "domain": "operations_sop", "city": None},
    {"query": "Trách nhiệm bồi thường của bên trông giữ xe khi xảy ra mất mát hư hỏng", "domain": "legal_policy", "city": None},
    {"query": "Quy chuẩn kỹ thuật bãi đỗ xe và hệ thống hạ tầng kỹ thuật", "domain": "technical_infra", "city": None},
]

print(f"✅ Đã tối ưu lại thành {len(df_rag)} chunks và chạy lại Benchmark:\n")
for i, item in enumerate(test_queries, start=1):
    print("="*80)
    print(f"🔎 TEST {i}: '{item['query']}'")
    res = retrieve_chunks_v2(item["query"], top_k=2, preferred_domain=item["domain"], city_filter=item["city"])
    for rank, (_, row) in enumerate(res.iterrows(), start=1):
        snippet = row["text"].replace("\n", " ")[:240]
        print(f"   🏆 Top {rank} [Score: {row['score']} | City: {row['city']}] | Nguồn: {row['source_file']} (Chunk {row['chunk_index']}/{row['total_chunks']})")
        print(f"      Nội dung: {snippet}...\n")

✅ Đã tối ưu lại thành 564 chunks và chạy lại Benchmark:

🔎 TEST 1: 'Giá gửi xe ô tô dưới 9 chỗ theo tháng ở khu vực Vành đai I Hà Nội'
   🏆 Top 1 [Score: 0.8421 | City: HN] | Nguồn: Bang_Gia_Do_Xe_O_To_HaNoi_TPHCM.xlsx (Chunk 3/12)
      Nội dung: [Nguồn: Bang_Gia_Do_Xe_O_To_HaNoi_TPHCM.xlsx | Nhóm: Bảng giá & Chính sách thương mại] ### Bảng dữ liệu (Sheet: 2_Ha_Noi_Gia_O_To) - BIỂU PHÍ TRÔNG GIỮ XE Ô TÔ TẠI HÀ NỘI: Bao gồm: Khung giá khu vực (Cũ vs Mới) & Chi tiết theo phân loại tải...

   🏆 Top 2 [Score: 0.7735 | City: HN] | Nguồn: Bang_Gia_Do_Xe_O_To_HaNoi_TPHCM.xlsx (Chunk 4/12)
      Nội dung: [Nguồn: Bang_Gia_Do_Xe_O_To_HaNoi_TPHCM.xlsx | Nhóm: Bảng giá & Chính sách thương mại] /01 giờ) - BIỂU PHÍ TRÔNG GIỮ XE Ô TÔ TẠI HÀ NỘI: STT | Unnamed: 1: Nội dung thu / Khu vực | Unnamed: 2: Xe ≤ 9 ghế ngồi / Xe tải ≤ 2 tấn | Unnamed: 3: X...

🔎 TEST 2: 'Xử lý thế nào khi khách hàng làm mất thẻ từ gửi xe hoặc gặp sự cố?'
   🏆 Top 1 [Score: 0.7832 | City: ALL] | Nguồn: Cẩm nang Xử lý Sự cố &

In [ ]:
# 1. Khởi tạo Gemini Client (Lấy API key miễn phí tại aistudio.google.com)
GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]  # export before starting Jupyter; never hard-code
client = genai.Client(api_key=GEMINI_API_KEY)

In [14]:
def ask_parking_assistant(user_query: str, preferred_domain: str = None, city_filter: str = None):
    # BƯỚC 1: RETRIEVAL - Tìm Top 3 đoạn tài liệu liên quan nhất từ kho 564 chunks
    top_chunks = retrieve_chunks_v2(
        query=user_query, 
        top_k=3, 
        preferred_domain=preferred_domain, 
        city_filter=city_filter
    )
    
    # Gom các chunks thành phần Ngữ cảnh (Context)
    context_blocks = []
    for idx, (_, row) in enumerate(top_chunks.iterrows(), start=1):
        context_blocks.append(f"--- TÀI LIỆU {idx} (Nguồn: {row['source_file']}) ---\n{row['text']}")
    context_str = "\n\n".join(context_blocks)

    # BƯỚC 2: AUGMENTATION - Lắp ráp System Prompt chống bịa đặt (Anti-Hallucination)
    system_instruction = """
    Bạn là Trợ lý AI Chăm sóc Khách hàng & Quản lý Vận hành Bãi đỗ xe chuyên nghiệp.
    Nhiệm vụ: Trả lời câu hỏi của người dùng một cách tự nhiên, rõ ràng, lịch sự và dễ hiểu.
    Quy tắc bắt buộc:
    1. CHỈ sử dụng thông tin được cung cấp trong phần [TÀI LIỆU THAM KHẢO].
    2. Diễn đạt lại các bảng biểu/điều luật khô khan thành câu văn mạch lạc, gạch đầu dòng rõ ràng.
    3. Luôn trích dẫn tên văn bản/tài liệu nguồn ở cuối câu trả lời.
    4. Nếu tài liệu không chứa thông tin để trả lời, hãy nói thẳng "Hiện tại hệ thống chưa có dữ liệu về vấn đề này", tuyệt đối không tự suy diễn.
    """

    prompt = f"""
    [TÀI LIỆU THAM KHẢO]:
    {context_str}

    [CÂU HỎI CỦA NGƯỜI DÙNG]:
    {user_query}
    """

    # BƯỚC 3: GENERATION - Gọi Gemini sinh câu trả lời tự nhiên
    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=0.2,
        )
    )
    return response.text, top_chunks[["source_file", "score"]]


In [15]:
cau_hoi = "Giá gửi xe ô tô dưới 9 chỗ theo tháng ở khu vực Vành đai I Hà Nội hiện nay là bao nhiêu?"
tra_loi, nguon = ask_parking_assistant(cau_hoi, preferred_domain="commercial_cs", city_filter="HN")
print(f"❓ CÂU HỎI: {cau_hoi}\n")
print(f"🤖 TRỢ LÝ AI TRẢ LỜI:\n{tra_loi}\n")
print("📚 Nguồn truy xuất được sử dụng:")
display(nguon)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}